# 02 — Предпросмотр аугментаций

Визуальная проверка трансформаций перед обучением.

In [ ]:
import sys; sys.path.insert(0, '..')
import cv2
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from src.data.transforms import build_train_transforms
from pathlib import Path

cfg = OmegaConf.merge(
    OmegaConf.load('../configs/base.yaml'),
    OmegaConf.load('../configs/unet.yaml'),
)

IMG_DIR = Path('../data/processed/images')
MASK_DIR = Path('../data/processed/masks')

transform = build_train_transforms(cfg)

In [ ]:
sample_path = sorted(IMG_DIR.glob('*.png'))[0]
img = cv2.cvtColor(cv2.imread(str(sample_path)), cv2.COLOR_BGR2RGB)
mask = cv2.imread(str(MASK_DIR / sample_path.name), cv2.IMREAD_GRAYSCALE)

n_aug = 8
fig, axes = plt.subplots(2, n_aug + 1, figsize=(22, 5))
axes[0, 0].imshow(img); axes[0, 0].set_title('Original'); axes[0, 0].axis('off')
axes[1, 0].imshow(mask, cmap='tab10', vmin=0, vmax=3); axes[1, 0].axis('off')

for i in range(1, n_aug + 1):
    aug = transform(image=img, mask=mask)
    # denormalize for display
    t = aug['image'].numpy().transpose(1, 2, 0)
    t = (t * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    axes[0, i].imshow(t); axes[0, i].set_title(f'Aug {i}'); axes[0, i].axis('off')
    axes[1, i].imshow(aug['mask'].numpy(), cmap='tab10', vmin=0, vmax=3); axes[1, i].axis('off')

plt.suptitle('Augmentation preview: images (top) and masks (bottom)')
plt.tight_layout()
plt.show()